# ⏱️ Aula 13 — Dinâmica de Processos e Séries Temporais

**Disciplina:** Inteligência Artificial Aplicada à Engenharia Química  
**Dataset:** `reator_dinamico.csv` — CSTR com dinâmica FOPDT (τ=10min, θ=5min)

---

## Contexto

Até agora os modelos tratavam cada linha como **independente**. Mas processos químicos são **dinâmicos**: a composição de saída responde a mudanças na vazão com **atraso** (θ) e **inércia** (τ). Precisamos capturar essa dinâmica.

## Modelo FOPDT (primeira ordem + atraso)

$$\tau \frac{dy}{dt} + y = K_p \, u(t - \theta)$$

- **τ** (constante de tempo): inércia do processo (CSTR: τ = V/F)
- **θ** (atraso de transporte): tempo do fluido na tubulação
- **Kp** (ganho): quanto a saída varia por unidade de entrada

## 3.1 — Exercício Guiado: Simulação + MLP com Lags

Simule o CSTR, identifique θ e τ, e treine uma MLP com lags.

### Passo 1: Importar e carregar

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

URL = "https://raw.githubusercontent.com/LuisGSVasconcelos/IA_EngQuimica/main/dados/aula13/reator_dinamico.csv"
df = pd.read_csv(URL, parse_dates=['timestamp'])
df.set_index('timestamp', inplace=True)
print(df.head())
print(df.describe().round(3))

### Passo 2: Visualizar série temporal + identificar θ e τ

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(df.index, df['F_alimentacao_L_min'], label='Vazão F')
axes[0].set_ylabel('F (L/min)'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(df.index, df['CA_mol_L'], color='steelblue', label='CA')
axes[1].set_ylabel('CA (mol/L)'); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_xlabel('Tempo')
plt.tight_layout()
plt.show()

# Você consegue ver os degraus? Onde a CA responde com atraso?

### Passo 3: Criar lags da vazão

In [ ]:
# Como a CA responde à F com atraso, usamos valores passados de F
for lag in [1, 5, 10, 15, 20]:
    df[f'F_lag{lag}'] = df['F_alimentacao_L_min'].shift(lag)
df = df.dropna()
print(f"Shape após lags: {df.shape}")

### Passo 4: Treinar MLP com lags

In [ ]:
features = [f'F_lag{l}' for l in [1, 5, 10, 15, 20]]
X = df[features]
y = df['CA_mol_L']

split = int(0.8 * len(X))
scaler = StandardScaler()
X_train = scaler.fit_transform(X.iloc[:split])
X_test = scaler.transform(X.iloc[split:])
y_train, y_test = y.iloc[:split], y.iloc[split:]

mlp = MLPRegressor(hidden_layer_sizes=(32, 16), max_iter=500, random_state=42)
mlp.fit(X_train, y_train)
y_pred = mlp.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"MLP + 5 lags RMSE: {rmse:.4f}")

### Passo 5: Plotar predição

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(df.index[split:], y_test, label='CA real')
plt.plot(df.index[split:], y_pred, '--', label=f'CA predito (MLP + 5 lags) RMSE={rmse:.4f}')
plt.legend(); plt.title('Predição dinâmica com MLP + lags')
plt.xlabel('Tempo'); plt.ylabel('CA (mol/L)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### ✏️ Pausa reflexiva (2 min)

Quantos lags você precisou para a MLP funcionar bem? Esse número é ~ θ + 3τ? (θ=5min, τ=10min → 5 + 30 = 35?)

> _Escreva aqui..._

---

## 3.2 — Exercício em Grupo: Quantos Lags?

Cada grupo testa um conjunto diferente de lags.

| Grupo | Lags | θ simulado |
|-------|------|-----------|
| **A** | [1, 2, 3] | 5 min |
| **B** | [1, 5, 10, 15, 20] | 5 min |
| **C** | [1, 5, 10, 15, 20, 30, 40] | 5 min |
| **D** | [1, 5, 10, 15, 20] | 15 min (atraso maior) |

In [ ]:
# Teste o conjunto de lags do seu grupo
lag_list = [1, 5, 10, 15, 20]   # ← mude para o do seu grupo

dfg = df.copy()
dfg = dfg[['CA_mol_L', 'F_alimentacao_L_min']].copy()
for lag in lag_list:
    dfg[f'F_lag{lag}'] = dfg['F_alimentacao_L_min'].shift(lag)
dfg = dfg.dropna()

Xg = dfg[[f'F_lag{l}' for l in lag_list]]
yg = dfg['CA_mol_L']
split = int(0.8 * len(Xg))
sc = StandardScaler()
Xtr = sc.fit_transform(Xg.iloc[:split]); Xte = sc.transform(Xg.iloc[split:])
m = MLPRegressor(hidden_layer_sizes=(32,16), max_iter=500, random_state=42)
m.fit(Xtr, yg.iloc[:split])
rmse_g = np.sqrt(mean_squared_error(yg.iloc[split:], m.predict(Xte)))
print(f"Lags {lag_list}: RMSE = {rmse_g:.4f}")

> **Perguntas:**
> 1. Qual grupo teve o menor RMSE?
> 2. A partir de quantos lags o ganho satura?
> 3. O grupo D (θ=15) capturou a dinâmica com lags até 20?

### 🧠 Desafio extra (NT)

Se θ = 30 min, você precisa de 30+ lags → MLP sofre com a maldição da dimensionalidade. É aí que a **LSTM** (Aula 14) ganha: memória interna que aprende quantos passos de história importam.

> _Escreva aqui..._

---

## Checklist

- [ ] Processo simulado/carregado
- [ ] θ e τ identificados visualmente
- [ ] Lags criados
- [ ] MLP treinada com lags
- [ ] RMSE calculado
- [ ] Número ótimo de lags encontrado